# Import das bibliotecas

In [1]:
import pandas as pd 
import numpy as np 
from datetime import datetime, timedelta

# Número de amostras e geração temporal

In [2]:
N = 1_365_000
inicio = datetime(2000, 1, 1)

timestamps = [inicio + timedelta(minutes=i*10) for i in range(N)]

In [3]:
timestamps

[datetime.datetime(2000, 1, 1, 0, 0),
 datetime.datetime(2000, 1, 1, 0, 10),
 datetime.datetime(2000, 1, 1, 0, 20),
 datetime.datetime(2000, 1, 1, 0, 30),
 datetime.datetime(2000, 1, 1, 0, 40),
 datetime.datetime(2000, 1, 1, 0, 50),
 datetime.datetime(2000, 1, 1, 1, 0),
 datetime.datetime(2000, 1, 1, 1, 10),
 datetime.datetime(2000, 1, 1, 1, 20),
 datetime.datetime(2000, 1, 1, 1, 30),
 datetime.datetime(2000, 1, 1, 1, 40),
 datetime.datetime(2000, 1, 1, 1, 50),
 datetime.datetime(2000, 1, 1, 2, 0),
 datetime.datetime(2000, 1, 1, 2, 10),
 datetime.datetime(2000, 1, 1, 2, 20),
 datetime.datetime(2000, 1, 1, 2, 30),
 datetime.datetime(2000, 1, 1, 2, 40),
 datetime.datetime(2000, 1, 1, 2, 50),
 datetime.datetime(2000, 1, 1, 3, 0),
 datetime.datetime(2000, 1, 1, 3, 10),
 datetime.datetime(2000, 1, 1, 3, 20),
 datetime.datetime(2000, 1, 1, 3, 30),
 datetime.datetime(2000, 1, 1, 3, 40),
 datetime.datetime(2000, 1, 1, 3, 50),
 datetime.datetime(2000, 1, 1, 4, 0),
 datetime.datetime(2000, 1, 1,

# Simulação da base com base nas colunas originais 

In [4]:
np.random.seed(42) 
umidade = np.clip(np.random.normal(40, 10, N), 10, 90) # distribuição centrada em 40%
ph = np.clip(np.random.normal(6.5, 0.7, N), 4.5, 8.5) # centrado em 6.5
n = np.random.choice([0, 1], size=N, p=[0.15, 0.85])
p = np.random.choice([0, 1], size=N, p=[0.15, 0.85])
k = np.random.choice([0, 1], size=N, p=[0.15, 0.85])
pop = np.round(np.random.uniform(0, 1, N), 2)
rain_3h = np.round(np.random.exponential(1.0, N), 2)

# Bloqueio meteorológico
wx_block = np.where((pop >= 0.6) | (rain_3h >= 2.0), 1, 0)

# Decisão da bomba
pump_state = np.where((umidade < 35) & (ph >= 6.0) & (ph <= 6.8) & (n==1) & (p==1) & (k==1) & (wx_block == 0), 1, 0)

In [5]:
pump_state

array([0, 0, 0, ..., 0, 0, 0], shape=(1365000,))

# DataFrame final

In [6]:
df = pd.DataFrame({
    'timestamp':timestamps, 
    'umidade': umidade, 
    'ph': ph, 
    'nitrogenio': n, 
    'fosforo':p, 
    'postassio': k, 
    'probabilidade_precipitacao': pop, 
    'chuva_3h': rain_3h, 
    'bloqueio_meteorológico': wx_block, 
    'estado_bomba': pump_state
})

In [7]:
df

,timestamp,umidade,ph,nitrogenio,fosforo,postassio,probabilidade_precipitacao,chuva_3h,bloqueio_meteorológico,estado_bomba
0,2000-01-01 00:00:00,44.967142,5.877727,1,1,1,0.36,0.43,0,0
1,2000-01-01 00:10:00,38.617357,6.035197,1,1,1,0.76,0.96,1,0
2,2000-01-01 00:20:00,46.476885,5.813679,1,1,1,0.86,0.13,1,0
3,2000-01-01 00:30:00,55.230299,5.647394,1,1,0,0.37,4.24,1,0
4,2000-01-01 00:40:00,37.658466,7.241370,1,0,1,0.47,0.54,0,0
...,...,...,...,...,...,...,...,...,...,...
1364995,2025-12-14 03:10:00,43.830865,6.541486,1,1,0,0.44,0.61,0,0
1364996,2025-12-14 03:20:00,38.106297,6.282486,1,0,1,0.05,3.65,1,0
1364997,2025-12-14 03:30:00,37.565239,6.716741,1,0,1,0.91,4.39,1,0
1364998,2025-12-14 03:40:00,40.533465,6.388291,1,0,0,0.36,0.11,0,0


# Salvar o DataFrame em CSV

In [8]:
df.to_csv('dados_irrigacao.csv', index=False)
df.sample(10)

,timestamp,umidade,ph,nitrogenio,fosforo,postassio,probabilidade_precipitacao,chuva_3h,bloqueio_meteorológico,estado_bomba
1263028,2024-01-06 00:40:00,43.365815,5.993094,0,1,1,0.14,0.68,0,0
385722,2007-05-02 15:00:00,42.281079,5.721081,1,1,1,0.55,0.69,0,0
720646,2013-09-13 11:40:00,15.233622,6.874455,1,1,1,0.60,0.40,1,0
710378,2013-07-04 04:20:00,28.933366,6.319996,1,1,1,0.38,0.52,0,1
523834,2009-12-16 17:40:00,35.747290,6.801352,1,1,1,0.91,0.01,1,0
148049,2002-10-25 02:50:00,33.084539,6.652774,1,0,1,0.53,0.09,0,0
968574,2018-06-01 05:00:00,38.029227,6.415791,0,0,1,0.94,0.83,1,0
1006640,2019-02-20 13:20:00,18.262001,7.511097,1,0,1,0.07,1.24,0,0
450675,2008-07-26 16:30:00,44.069580,5.104972,1,1,0,0.60,1.18,1,0
459796,2008-09-28 00:40:00,49.544820,6.676021,1,1,1,0.41,1.00,0,0
